In [178]:
import bffile
from pathlib import Path
import nd2
import numpy as np
import tifffile as tifff

In [158]:
#img_path=Path(r"Y:\Ti2-W1\Kim Boonekamp\KB_2026_041_R3_myh9_gfp\raw_data\KB_2026_0408_07myh9-002_crop.nd2")
img_path=Path(r"D:\VictorPerez\repos\KB_2026_0408_07myh9-001.nd2")

In [159]:
metadata={"series_count":None,"resolutions":None,"ome":None,"core":None}
with bffile.BioFile(img_path) as bf:
    metadata["series_count"]=bf.series_count()
    metadata["ome"]=bf.ome_metadata
    metadata["core"]=bf.core_metadata()


metadata["series"]=[]
for i in range(metadata["series_count"]):
    metadata["series"].append({"id":i,
                               "x":metadata["ome"].images[i].pixels.planes[0].position_x,
                               "y":metadata["ome"].images[i].pixels.planes[0].position_y,
                               "channels":[ ch.name for ch in metadata["ome"].images[i].pixels.channels]
                               }
                               )

In [162]:
metadata["ome"].images

[Image(
    id='Image:0',
    name='KB_2026_0408_07myh9-001.nd2 (series 01)',
    instrument_ref={'id': 'Instrument:0'},
    objective_settings={'id': 'Objective:0:0', 'refractive_index': 1.333},
    pixels={'channels': [{'detector_settings': {'id': 'Detector:0:0'}, 'light_path': {}, 'id': 'Channel:0:0', 'name': 'FITC', 'samples_per_pixel': 1, 'emission_wavelength': 525.0, 'color': Color('#24ff00', rgb=(36, 255, 0))}, {'detector_settings': {'id': 'Detector:0:0'}, 'light_path': {}, 'id': 'Channel:0:1', 'name': 'bf', 'samples_per_pixel': 1}], 'metadata_only': {}, 'planes': [{'the_z': 0, 'the_t': 0, 'the_c': 0, 'delta_t': 5.282799324999563, 'exposure_time': 0.35, 'position_x': -10417.5, 'position_x_unit': <UnitsLength.MICROMETER: 'µm'>, 'position_y': -6750.900000000001, 'position_y_unit': <UnitsLength.MICROMETER: 'µm'>, 'position_z': 4932.72, 'position_z_unit': <UnitsLength.MICROMETER: 'µm'>}, {'the_z': 0, 'the_t': 0, 'the_c': 1, 'delta_t': 5.282799324999563, 'exposure_time': 0.1, 'positi

In [191]:
def write_tiff(img_array,channel,ndims,outdir,file_name):
    outfile = outdir / f"{file_name}.ome.tif"
    no_of_planes=ndims["planes"]
    no_of_frames=ndims["frames"]
    with tifff.TiffWriter(outfile, bigtiff=True, ome=False) as tif:
        for t in range(no_of_frames):
            for z in range(no_of_planes):                
                tif.write(
                    data=img_array[t,channel,z,:,:],
                    subfiletype=0,
                    subifds=None,
                    photometric="minisblack",
                    )
    return outfile

In [226]:
def write_tiff2(lazy_array,channel,ndims,outdir,file_name):
    outfile = outdir / f"{file_name}.ome.tif"
    no_of_planes=ndims["planes"]
    no_of_frames=ndims["frames"]
    print(lazy_array.shape)
    with tifff.TiffWriter(outfile, bigtiff=True, ome=False) as tif:
        for t in range(no_of_frames):
            for z in range(no_of_planes):       
                tif.write(
                    data= np.array(lazy_array[t,channel,z,:,:]) ,
                    dtype=lazy_array.dtype.name,
                    subfiletype=0,
                    subifds=None,
                    photometric="minisblack",
                    )
    return outfile

In [180]:
outdir=Path(r"D:\VictorPerez\repos\nd2_splitter\output")
with bffile.BioFile(img_path) as bf:
    for i in range(2):
        lazy_array = bf.as_array(series=i,resolution=0)
        output_file=f"{img_path.stem}_xy{i:03d}.tif"
        

In [181]:
lazy_array.shape

(1, 2, 31, 1024, 1024)

In [227]:
bf=bffile.BioFile(img_path)
bf.open()
lazy_array = bf.as_array(series=0,resolution=0)
ndims={"planes":int(31),"frames":int(1)}
output_file=f"{img_path.stem}_xy{0:03d}.tif"
write_tiff2(lazy_array,0,ndims,outdir,output_file)
bf.close()

(1, 2, 31, 1024, 1024)


In [ ]:
lazy_array.take

(1024, 1024)

In [157]:
metadata["ome"].images[0].pixels.planes

[Plane(
    the_z=0,
    the_t=0,
    the_c=0,
    delta_t=11.11097127499804,
    exposure_time=0.35,
    position_x=-10368.7,
    position_x_unit='µm',
    position_y=-6489.0,
    position_y_unit='µm',
    position_z=4930.12,
    position_z_unit='µm',
 ),
 Plane(
    the_z=0,
    the_t=0,
    the_c=1,
    delta_t=11.11097127499804,
    exposure_time=0.1,
    position_x=-10368.7,
    position_x_unit='µm',
    position_y=-6489.0,
    position_y_unit='µm',
    position_z=4930.12,
    position_z_unit='µm',
 ),
 Plane(
    the_z=1,
    the_t=0,
    the_c=0,
    delta_t=11.611851274998859,
    exposure_time=0.35,
    position_x=-10368.7,
    position_x_unit='µm',
    position_y=-6489.0,
    position_y_unit='µm',
    position_z=4932.1,
    position_z_unit='µm',
 ),
 Plane(
    the_z=1,
    the_t=0,
    the_c=1,
    delta_t=11.611851274998859,
    exposure_time=0.1,
    position_x=-10368.7,
    position_x_unit='µm',
    position_y=-6489.0,
    position_y_unit='µm',
    position_z=4932.1,
    